# 前後の単語を考慮してかな漢字変換をする（bigram 言語モデル）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/31_japanese_input/kkc_bigram.ipynb)

NLP クックブック 31-05 ／ 日本語入力（かな漢字変換）

同音異義語を文脈に合わせて選びたい。直前の単語との並びやすさ (bigram) を使って変換する。

In [1]:
!pip install -q fugashi unidic-lite jaconv marisa-trie

In [2]:
import io, math, re, urllib.request, zipfile
from collections import Counter, defaultdict
import jaconv
import marisa_trie
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/000148/files/"
sentences = []
for f in ["789_ruby_5639.zip", "752_ruby_2438.zip", "773_ruby_5968.zip"]:
    for s in re.split(r"(?<=。)|\n", aozora(base + f)):
        if s.strip():
            sentences.append([(w.surface, jaconv.kata2hira(w.feature.kana or w.surface)) for w in tagger(s.strip())])
train, test = sentences[:-300], sentences[-300:]

dic, unigram, bigram = defaultdict(set), Counter(), Counter()
for sent in train:
    prev = "<s>"
    unigram[prev] += 1
    for surface, reading in sent:
        dic[reading].add(surface)
        unigram[surface] += 1
        bigram[prev, surface] += 1
        prev = surface
    bigram[prev, "</s>"] += 1
total, trie = sum(unigram.values()), marisa_trie.Trie(dic.keys())
print(len(train), "文で学習 /", len(unigram), "語 /", len(bigram), "種類の bigram")

16659 文で学習 / 17499 語 / 102445 種類の bigram


In [3]:
LAMBDA, UNKNOWN_COST = 0.8, 20.0

def cost(prev, word, lam=LAMBDA):
    """−log P(word | prev)。bigram と unigram を混ぜる (線形補間) ことで、見たことのない並びにも確率を与える"""
    p_uni = unigram.get(word, 0) / total
    p_bi = bigram.get((prev, word), 0) / unigram[prev] if unigram.get(prev) else 0.0
    p = lam * p_bi + (1 - lam) * p_uni
    return -math.log(p) if p > 0 else UNKNOWN_COST

def convert(kana, lam=LAMBDA):
    # best[位置][その位置で終わる単語] = (コスト, 直前の位置, 直前の単語)。直前の単語ごとに最小コストを覚えておく
    best = [dict() for _ in range(len(kana) + 1)]
    best[0]["<s>"] = (0.0, None, None)
    for i in range(len(kana)):
        if not best[i]:
            continue
        candidates = [(i + len(r), s) for r in trie.prefixes(kana[i:]) for s in dic[r]] + [(i + 1, kana[i])]
        for j, word in candidates:
            for prev, (c, _, _) in best[i].items():
                new = c + cost(prev, word, lam)
                if word not in best[j] or new < best[j][word][0]:
                    best[j][word] = (new, i, prev)
    word = min(best[-1], key=lambda w: best[-1][w][0] + cost(w, "</s>", lam))
    words, pos = [], len(kana)
    while pos > 0:
        _, prev_pos, prev = best[pos][word]
        words.append(word); pos, word = prev_pos, prev
    return words[::-1]

for kana in ["あめがふってきた", "かみをきった", "ひをつける", "はをみがく", "かぜをひく", "きがつく", "はながさく", "はしでたべる", "はしをわたる", "くすりがきく"]:
    print(f"{kana} → unigram: {''.join(convert(kana, lam=0.0))} / bigram: {''.join(convert(kana))}")

あめがふってきた → unigram: 雨が振っ的た / bigram: 雨が降って来た
かみをきった → unigram: 神を切った / bigram: 髪を切った
ひをつける → unigram: 日をつける / bigram: 火をつける
はをみがく → unigram: はを磨く / bigram: 歯を磨く
かぜをひく → unigram: 風を引く / bigram: 風邪を引く
きがつく → unigram: 来がつく / bigram: 気がつく
はながさく → unigram: 鼻が作 / bigram: 鼻が咲く
はしでたべる → unigram: はしで食べる / bigram: 箸で食べる
はしをわたる → unigram: はしを渡る / bigram: 箸を渡る
くすりがきく → unigram: 薬が聞く / bigram: 薬が聞く


In [4]:
# 学習に使っていない 300 文で、文がまるごと正しく変換できた割合を比べる
def sentence_accuracy(lam, n=300):
    ok = 0
    for sent in test[:n]:
        kana, gold = "".join(r for _, r in sent), "".join(s for s, _ in sent)
        ok += "".join(convert(kana, lam)) == gold
    return ok / n

for lam in [0.0, 0.5, 0.8, 0.95]:
    print(f"lambda = {lam}: 文正解率 {sentence_accuracy(lam):.3f}")

lambda = 0.0: 文正解率 0.170
lambda = 0.5: 文正解率 0.323
lambda = 0.8: 文正解率 0.317
lambda = 0.95: 文正解率 0.313


- bigram では、「髪を → 切っ」「火を → つける」「風邪を → 引く」のように、直前の単語との並びやすさで表記が選ばれる。unigram では区別できなかった同音異義語が、文脈で決まるようになる。
- 直前の 1 語しか見ないので、「はしを わたる」の「はし」は後ろの「渡る」を手がかりにできず「箸」になる。学習データにない並び（「薬が効く」）も当てられない。
- ビタビアルゴリズムの状態が「位置」から「位置と、そこで終わる単語」の組に増える。計算量は候補数の 2 乗に比例するが、文が短いので実用上は問題ない。
- 学習データにない単語の並びの確率が 0 にならないよう、unigram と混ぜる（平滑化）。混ぜる割合は、学習に使っていないデータで決める。
- 実用の変換エンジン（Mozc など）は、単語そのものではなく品詞や単語クラスの bigram を使う。単語 bigram より少ないデータで学習でき、メモリも小さくて済む。
- 文正解率は厳しい指標（1 か所でも違えば不正解）。部分的な正しさも測る指標は、この章の評価のレシピを参照。
- 単語ラティス上の確率的かな漢字変換や Viterbi アルゴリズム、Pointwise 予測・CRF・MEMM の比較を扱う演習は、姉妹編『[自然言語処理の教科書](https://github.com/mamorlis/nlpbook)』第 3 章の [単語ラティス・仮名漢字変換・CRF vs MEMM・Pointwise 予測](https://github.com/mamorlis/nlpbook/blob/main/notebooks/ch3_viterbi_crf.ipynb)（確率的仮名漢字変換と Pointwise 予測を含む）も参照。